# 05 · Supplier Analysis

Full supplier-level report: revenue history, margin, stock exposure, activity classification, collection breakdown, and reorder requirements.

**Visuals are included** — supplier reports go to management and need charts. All charts use static Plotly (no ipywidgets) so they render in GitHub's notebook viewer without running the code.

```
inventory-analysis/
├── notebooks/  05_supplier_analysis.ipynb
├── src/        supplier_analysis.py
└── outputs/    Supplier_<name>_YYYYMMDD.xlsx
```

## 0 · Dependencies

In [ ]:
!pip install pandas numpy plotly openpyxl -q

## 1 · Imports

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import supplier_analysis as sa
print('imports ok')

imports ok


## 2 · Load data

In [2]:
combined_df = pd.read_parquet(PROJECT_ROOT / 'outputs' / 'combined_df.parquet')

# Optional: load forecast output from notebook 02
# from src.inventory_forecast import run_demand_forecast
# forecast_df, _ = run_demand_forecast(combined_df)
forecast_df = None

print('Available suppliers:')
print(combined_df['Supplier'].value_counts().head(15).to_string())

Available suppliers:
Supplier
D'DECOR HOME FABRICS PVT.LTD                677011
ZHEJIANG FAMOUS TEXTILE CO., LTD.           616368
D'DECOR EXPORTS PVT. LTD.                   587620
ZHEJIANG SITAIBAO TEXTILE CO., LTD-YAYI     438564
G.M.FABRICS PVT.LTD                         198831
J.W.L. MAINWAY CO., LTD                     162422
DICITEX FURNISHING PVT .LTD                 157778
SHAOXING XIAOXUANCHUANG H.H FAB CO. LTD.    157568
G.M. SYNTEX PVT.LTD                         118297
HAINING HUANYU WARP KNITTING CO.,LTD.        95473
HANGZHOU E-WAY INTERNATIONAL TRADE CO.       82139
CASTILLA TEXTIL.                             78289
SHAOXING KEQIAO KANGWO TEXTILE CO., LTD      65777
Ekart Tekstil Sanayi ve Ticaret A.S.         57944
HANGZHOU ZHONGYI FABRIC CO.                  43784


## 3 · Run analysis

Set `SUPPLIER` to any name from the list above.

In [3]:
SUPPLIER = combined_df['Supplier'].value_counts().index[0]  # largest supplier by default

result = sa.run_supplier_analysis(
    combined_df,
    supplier_name=SUPPLIER,
    forecast_df=forecast_df,
    adjusted_cost_path=None,   # optional: path to revised cost Excel
    output_dir=str(PROJECT_ROOT / 'outputs'),
)

sku_df   = result['sku_df']
coll_df  = result['collection_df']
raw_df   = result['supplier_df']
current_year = pd.Timestamp.now().year


── Supplier Analysis  ·  D'DECOR HOME FABRICS PVT.LTD ──────────────────────
  records    : 677,011
  SKUs       : 9,650
  date range : 2019-01-01 → 2025-12-20

  Building SKU table
  Building collection table

  Saving Excel
  saved  ·  Supplier_DDECOR HOME FABRICS PVTLTD_20260315_024703.xlsx

── Summary ───────────────────────────────────────────────────
  SKUs           : 9,650
  Collections    : 250
  Very Active    : 0
  Slow/Inactive  : 0
  Stock Value    : 0 SAR


## 4 · Dashboards

### 4.1 · Revenue timeline

**What you're looking at:** Monthly revenue and quantity trend for this supplier. The dual-axis view catches the most important signal: if revenue and quantity diverge, the average selling price is shifting — either the product mix is changing or pricing has moved.

**Action:** A sustained gap where revenue grows faster than quantity = price increases are holding. Revenue declining while quantity holds = you're selling more but earning less per unit.

In [5]:
monthly = (
    raw_df.assign(YM=raw_df['Date'].dt.to_period('M'))
    .groupby('YM')
    .agg(Revenue=('bal Value','sum'), Qty=('bal Qty','sum'))
    .reset_index()
)
monthly['YM'] = monthly['YM'].astype(str)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=monthly['YM'], y=monthly['Revenue'],
    name='Revenue (SAR)', marker_color='#1f77b4', opacity=0.7,
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=monthly['YM'], y=monthly['Qty'],
    name='Quantity', mode='lines+markers',
    line=dict(color='#ff7f0e', width=2.5), marker=dict(size=5),
), secondary_y=True)
fig.update_layout(
    title=f'{SUPPLIER}  ·  Monthly Revenue & Quantity',
    height=450, plot_bgcolor='black', paper_bgcolor='black',
    xaxis_tickangle=-45, legend=dict(orientation='h', y=1.1),
)
fig.update_yaxes(title_text='Revenue (SAR)', secondary_y=False)
fig.update_yaxes(title_text='Quantity',      secondary_y=True)
fig.show()

### 4.2 · Year-over-year comparison by collection

**What you're looking at:** Each bar group is a collection. Current year revenue vs prior year, side by side. Collections are sorted by current-year revenue.

**Action:** Collections with a shrinking bar are losing relevance. Collections with a growing bar should be prioritised for replenishment. Missing prior-year bar = the collection was introduced this year.

In [7]:
cy_col = f'Sales_{current_year}'
py_col = f'Sales_{current_year - 1}'

yoy_cols = ['Section'] + [c for c in [cy_col, py_col] if c in coll_df.columns]
yoy = coll_df[yoy_cols].nlargest(20, cy_col).copy() if cy_col in coll_df.columns else coll_df.head(20)

fig = go.Figure()
if cy_col in yoy.columns:
    fig.add_trace(go.Bar(
        x=yoy['Section'].astype(str), y=yoy[cy_col],
        name=str(current_year), marker_color='#2ca02c',
    ))
if py_col in yoy.columns:
    fig.add_trace(go.Bar(
        x=yoy['Section'].astype(str), y=yoy[py_col],
        name=str(current_year - 1), marker_color='#aec7e8',
    ))
fig.update_layout(
    title=f'YoY Revenue  ·  Top 20 Collections  ·  {current_year} vs {current_year-1}',
    barmode='group', height=500,
    xaxis_tickangle=-30, plot_bgcolor='black', paper_bgcolor='black',
    yaxis_title='Revenue (SAR)',
)
fig.show()

In [10]:
combined_df.columns

Index(['Section', 'Section Name', 'bal Value', 'bal Qty', 'U Price', 'Client',
       'Outlet', 'Outlet Name', 'Date', 'SKU', 'Catalog No.', 'Year',
       'Total_Cost', 'Total_Profit', 'Profit_Margin_%', 'CURRENT_STOCK',
       'OUTSTANDING', 'PR', 'EFFECTIVE_STOCK', 'CATEGORY', 'Supplier',
       'LAST_ENTRY_DATE', 'Nb. Days (Avail. Balance)', 'COST_PER_DOLLAR',
       'COLOR_NAME', 'ARTPATERN', 'TEXTURE', 'SUB_CATEGORY',
       'IS_PLAIN_SECTION', 'CATALOG_NO', 'COLLECTION_NAME', 'SERIAL',
       'SKU-STATUES-Mahmoud', 'SKU STATUES -Python', 'OLD_QTY',
       'First_Inv_Date', 'Outlet_Type', 'Area_Master_Category', 'Country',
       'Sales_Type'],
      dtype='str')

### 4.3 · Margin vs stock turnover — SKU portfolio map

**What you're looking at:** Every bubble is an SKU. **X** = margin %, **Y** = stock turnover ratio (current-year sales / stock value). Bubble size = current-year revenue. Color = activity level.

**Four quadrants:**

| | Low Turnover | High Turnover |
|---|---|---|
| **High Margin** | Good profit, slow stock movement — reorder conservatively | Best position: profitable *and* moving fast |
| **Low Margin**  | Worst position: low margin *and* slow — review pricing or discontinue | High velocity but thin margin — check if volume justifies cost |

**Action:** Large bubbles in the bottom-left are the highest-priority pricing reviews.

In [12]:
plot = sku_df[
    sku_df['Profit_Margin%'].between(-30, 70) &
    sku_df['Stock_Turnover'].between(0, 20)
].copy()

cy_col = f'Sales_{current_year}'
size_col = cy_col if cy_col in plot.columns else 'Stock_Value'

fig = px.scatter(
    plot,
    x='Margin_%', y='Stock_Turnover',
    size=plot[size_col].clip(lower=1), size_max=35,
    color='Activity',
    color_discrete_map={
        'Very Active':'#2ca02c','Active':'#1f77b4',
        'Slow Moving':'#ff7f0e','Inactive':'#d62728','Unknown':'#aaa',
    },
    hover_data=['SKU','Section','CATEGORY','Stock_Value','Days_Since_Last_Sale'],
    title=f'{SUPPLIER}  ·  SKU Portfolio Map  (bubble = {current_year} revenue)',
    labels={'Margin_%':'Margin %','Stock_Turnover':'Stock Turnover (x)'},
)
fig.add_vline(x=0,  line_dash='dash', line_color='red',   annotation_text='Break-even')
fig.add_vline(x=25, line_dash='dot',  line_color='green', annotation_text='25% target')
fig.add_hline(y=2,  line_dash='dot',  line_color='grey',  annotation_text='2x turnover')
fig.update_layout(height=600, plot_bgcolor='black', paper_bgcolor='black')
fig.show()

KeyError: 'Profit_Margin%'

### 4.4 · Activity breakdown by collection

**What you're looking at:** A stacked bar per collection showing how many SKUs fall into each activity level. The total bar height = total SKUs in the collection.

**How to read it:** A collection that is mostly inactive or slow-moving is a dead-weight collection — capital is tied up in unsold stock with no velocity. A collection that is mostly Very Active has strong demand momentum.

**Action:** Collections with > 50% inactive SKUs should be reviewed for clearance, discontinuation, or consolidated ordering.

In [14]:
act = (
    sku_df.groupby(['Section','Activity'])
    .size()
    .reset_index(name='Count')
)
top_sections = sku_df.groupby('Section').size().nlargest(20).index
act = act[act['Section'].isin(top_sections)]

color_map = {
    'Very Active':'#2ca02c','Active':'#1f77b4',
    'Slow Moving':'#ff7f0e','Inactive':'#d62728','Unknown':'#aaa',
}
fig = px.bar(
    act,
    x='Section', y='Count', color='Activity',
    color_discrete_map=color_map,
    barmode='stack',
    title=f'{SUPPLIER}  ·  SKU Activity by Collection  (top 20)',
    labels={'Section':'Collection','Count':'SKU Count'},
    category_orders={'Activity':['Very Active','Active','Slow Moving','Inactive']},
)
fig.update_layout(height=500, xaxis_tickangle=-30,
                  plot_bgcolor='black', paper_bgcolor='black')
fig.show()

### 4.5 · Stock exposure map

**What you're looking at:** Collections sorted by stock value (SAR tied up in inventory). Bar color shows the stock-to-sales ratio: how many times over you are stocked relative to current-year sales.

**How to read it:** Dark red = you have far more stock than you sold this year. Dark green = stock is lean relative to sales velocity. The goal is to have the tallest bars (high stock value) also be green (turning fast).

**Action:** Tall red bars are the most urgent capital efficiency problem — high stock value that is not generating proportional revenue.

In [16]:
stock_exp = sku_df.groupby('Section').agg(
    Stock_Value  = ('Stock_Value', 'sum'),
    SKU_Count    = ('SKU',         'count'),
).reset_index()

cy_col = f'Sales_{current_year}'
if cy_col in sku_df.columns:
    cy_sales = sku_df.groupby('Section')[cy_col].sum().reset_index()
    stock_exp = stock_exp.merge(cy_sales, on='Section', how='left')
    stock_exp['Stock_Sales_Ratio'] = np.where(
        stock_exp[cy_col] > 0,
        stock_exp['Stock_Value'] / stock_exp[cy_col], 99)
    color_col = 'Stock_Sales_Ratio'
else:
    stock_exp['Stock_Sales_Ratio'] = 0
    color_col = 'Stock_Value'

stock_exp = stock_exp.nlargest(20, 'Stock_Value')

fig = px.bar(
    stock_exp,
    x='Section', y='Stock_Value',
    color='Stock_Sales_Ratio',
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=1.0,
    hover_data=['SKU_Count', 'Stock_Sales_Ratio'],
    text=stock_exp['Stock_Value'].apply(lambda x: f'{x/1000:.0f}K'),
    title=f'{SUPPLIER}  ·  Stock Exposure  (color = stock / {current_year} sales)',
    labels={'Stock_Value':'Stock Value (SAR)','Section':'Collection',
            'Stock_Sales_Ratio':'Stock/Sales'},
)
fig.update_traces(textposition='outside')
fig.update_layout(height=520, xaxis_tickangle=-30,
                  plot_bgcolor='black', paper_bgcolor='black')
fig.show()

## 5 · Summary tables

### Collection summary

In [17]:
from IPython.display import display

coll_cols = ['Section','SKU_Count','Total_Sales','Total_Qty','Total_Profit',
             'Margin_%','YoY_Growth_%',
             f'Sales_{current_year}', f'Sales_{current_year-1}']
present   = [c for c in coll_cols if c in coll_df.columns]
fmt       = {c: '{:,.0f}' for c in present if 'Sales' in c or c in ['Total_Sales','Total_Qty','Total_Profit']}
fmt.update({'Margin_%': '{:.1f}%', 'YoY_Growth_%': '{:+.1f}%'})
fmt = {k: v for k, v in fmt.items() if k in present}

styled = coll_df[present].head(20).style.format(fmt)
if 'Margin_%' in present:
    styled = styled.background_gradient(subset=['Margin_%'], cmap='RdYlGn', vmin=-20, vmax=40)
if 'YoY_Growth_%' in present:
    styled = styled.background_gradient(subset=['YoY_Growth_%'], cmap='RdYlGn', vmin=-50, vmax=50)
display(styled)

,Section,SKU_Count,Total_Sales,Total_Qty,Total_Profit,Margin_%,YoY_Growth_%,Sales_2026,Sales_2025
192,4932,89,"8,251,643","195,444","5,717,968",69.3%,-100.0%,nan,"207,432"
143,4354,220,"7,579,782","182,767","5,664,557",74.7%,-100.0%,nan,"403,980"
221,5107,52,"7,533,106","135,453","4,664,833",61.9%,-100.0%,nan,"556,092"
184,4913,140,"6,745,606","152,093","3,867,544",57.3%,-100.0%,nan,"1,071,140"
157,4658,68,"6,531,352","157,996","4,177,276",64.0%,-100.0%,nan,"243,568"
206,5021,112,"5,761,611","121,058","3,861,647",67.0%,-100.0%,nan,"373,033"
217,5098,90,"5,571,990","123,117","3,576,176",64.2%,-100.0%,nan,"394,162"
230,5217,25,"5,542,135","128,321","2,969,152",53.6%,-100.0%,nan,"2,514,324"
197,4938,72,"5,094,396","131,360","3,399,245",66.7%,-100.0%,nan,"34,373"
185,4914,106,"4,817,342","111,246","2,707,010",56.2%,-100.0%,nan,"630,798"


### Top & bottom SKUs by margin

In [18]:
cols = ['SKU','Section','CATEGORY','Activity','Margin_%','Stock_Turnover',
        'Stock_Value','Days_Since_Last_Sale','YoY_Growth_%']
if 'Reorder_Qty' in sku_df.columns: cols.append('Reorder_Qty')
present = [c for c in cols if c in sku_df.columns]
fmt = {'Margin_%':'{:.1f}%','Stock_Turnover':'{:.2f}x',
       'Stock_Value':'{:,.0f}','YoY_Growth_%':'{:+.1f}%',
       'Days_Since_Last_Sale':'{:.0f}','Reorder_Qty':'{:,}'}
fmt = {k: v for k, v in fmt.items() if k in present}

valid = sku_df[sku_df['Margin_%'].between(-50, 80)]

print('── Top 15 SKUs by Margin ──────────────────────────────────────')
top = valid.nlargest(15, 'Margin_%')[present]
display(top.style.format(fmt).background_gradient(
    subset=['Margin_%'], cmap='RdYlGn', vmin=-20, vmax=40))

print('── Bottom 15 SKUs by Margin ───────────────────────────────────')
bot = valid.nsmallest(15, 'Margin_%')[present]
display(bot.style.format(fmt).background_gradient(
    subset=['Margin_%'], cmap='RdYlGn', vmin=-20, vmax=40))

KeyError: 'Margin_%'

### Reorder required

In [19]:
if 'Reorder_Qty' in sku_df.columns:
    rcols = ['SKU','Section','CATEGORY','Activity',
             'Stock_Value','Days_Coverage','Reorder_Qty','Monthly_Demand_Final']
    present = [c for c in rcols if c in sku_df.columns]
    reorder = sku_df[sku_df['Reorder_Qty'] > 0][present].sort_values('Reorder_Qty', ascending=False)
    print(f'Reorder required: {len(reorder):,} SKUs  |  '
          f'Total qty: {reorder["Reorder_Qty"].sum():,.0f}')
    rfmt = {'Reorder_Qty':'{:,}','Stock_Value':'{:,.0f}',
            'Monthly_Demand_Final':'{:.1f}','Days_Coverage':'{:.0f}'}
    rfmt = {k: v for k, v in rfmt.items() if k in present}
    display(
        reorder.head(20).style.format(rfmt)
        .background_gradient(subset=['Reorder_Qty'], cmap='Reds')
    )
else:
    print('Run notebook 02 first and pass forecast_df to get reorder quantities.')

Run notebook 02 first and pass forecast_df to get reorder quantities.


## 6 · Output file

The Excel report contains:

| Sheet | Contents |
|-------|----------|
| Summary | Key metrics at a glance |
| SKU Analysis | Full per-SKU table with all yearly pivots |
| Collections | Collection-level revenue and margin summary |
| Reorder Required | SKUs with Reorder_Qty > 0 (if forecast provided) |
| Slow & Inactive | SKUs classified as Slow Moving or Inactive |


In [ ]:
print(f'Output : {result["excel_path"]}')
print(f'Size   : {result["excel_path"].stat().st_size / 1024:.1f} KB')